In [3]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output


# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import base64
from PythonCRUD import AnimalShelter

image_path = './Grazioso_Salvare_Logo.png'

with open(image_path, 'rb') as f:
    image_base64 = base64.b64encode(f.read()).decode('ascii')

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU1234"
shelter = AnimalShelter(username, password)

df = pd.DataFrame.from_records(shelter.read({}))
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.Img(src=f'data:image/png;base64,{image_base64}',
                         style={'height': '150px'})),
    html.Center(html.B(html.H1('Grazioso Salvare Rescue Animal Dashboard'))),
    html.Hr(),
    #Drop down menu for filtering
    dcc.Dropdown(
        id='filter-dropdown',
        options=[
            {'label': 'Water Rescue', 'value': 'water_rescue'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain_rescue'},
            {'label': 'Disaster Rescue/Tracking', 'value': 'disaster_rescue'},
            {'label': 'Reset', 'value': 'reset'}
        ],
        placeholder='Select a filter'
    ),
    html.Br(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Table Features
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'}
    ),
    html.Br(),
    html.Hr(),
    html.Div([
        html.Div(id='map-id', style={'width': '45%', 'display': 'inline-block'}),
        html.Div(dcc.Graph(id='pie-chart-id'), style={'width': '45%', 'display': 'inline-block'})
    ], style={'display': 'flex'}),
    html.P("Developer: Nathaniel Lemons SNHU-340")
])

#############################################
# Interaction Between Components / Controller
#############################################
#Update table columns and data based on filter selection
@app.callback(
    [Output('datatable-id', 'columns'),
    Output('datatable-id', 'data')],
    [Input('filter-dropdown', 'value')]
)
def update_table(filter_value):
    query = {}
    if filter_value == 'water_rescue':
        query = {"$and": [
            {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}},
            {"sex_upon_outcome": "Intact Female"},
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
        ]}
    elif filter_value == 'mountain_rescue':
        query = {"$and": [
            {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdop", "Siberian Husky", "Rottweiler"]}},
            {"sex_upon_outcome": "Intact Male"},
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 256}}
        ]}
    elif filter_value == 'disaster_rescue':
        query = {"$and": [
            {"breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}},
            {"sex_upon_outcome": "Intact Male"},
            {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
        ]}
    elif filter_value == 'reset':
        query = {}
        
    df = pd.DataFrame.from_records(shelter.read(query))
    if not df.empty:
        df.drop(columns=['_id'], inplace=True)
        columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
        data = df.to_dict('records')
    else:
        columns = []
        data = []
        
    return columns, data

#This callback will highlight a row on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    map_style = {'width': '100%', 'height': '500px'}
    if index:
        dff = pd.DataFrame.from_dict(viewData)
        row = index[0]

        latitude = dff.iloc[row, dff.columns.get_loc('location_lat')]
        longitude = dff.iloc[row, dff.columns.get_loc('location_long')]

        return [
            dl.Map(style=map_style,
                   center=[latitude, longitude], zoom=10, children=[
                       dl.TileLayer(id="base-layer-id"),
                       dl.Marker(position=[latitude, longitude],
                                 children=[
                                     dl.Tooltip(dff.iloc[row, dff.columns.get_loc('breed')]),
                                     dl.Popup([
                                         html.H1("Animal Name"),
                                         html.P(dff.iloc[row, dff.columns.get_loc('name')])
                                     ])
                                 ])
                   ])
        ]
    else:
        return [dl.Map(style=map_style, center=[30.2672, -97.7431], zoom=10, children=[dl.TileLayer(id="base-layer-id")])]
    
#Update pie chart based on filter selection
@app.callback(
    Output('pie-chart-id', 'figure'),
    [Input('datatable-id', 'data')]
)
def update_pie_chart(data):
    if data:
        dff = pd.DataFrame(data)
        fig = px.pie(dff, names='breed', title='Distribution of Breeds')
        fig.update_layout(width=700, height=500)
        return fig
    else:
        fig = px.pie(title='No Data')
        fig.update_layout(width=700, height=500)
        return fig
    
app.run_server(debug=True)

Dash app running on http://127.0.0.1:10891/
